# 2024C 农作物种植策略：整数规划与情景审计

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "请从仓库内运行 Notebook"


## 1. 官方附件读取

In [ ]:
from cumcm_lens.cases.crop_planning import locate_attachments, load_official_data
files = locate_attachments(ROOT / "data/raw")
land, crops, planted, stats, demand, quality = load_official_data(
    files["附件1.xlsx"], files["附件2.xlsx"])
quality


## 2. 数据字典与销量口径

In [ ]:
display(land.groupby("land_type").agg(plots=("plot","size"), area_mu=("area_mu","sum")))
display(demand.head(10))


## 3. 朴素基线及约束失败

In [ ]:
from cumcm_lens.cases.crop_planning import repeated_2023_baseline
baseline, baseline_audit = repeated_2023_baseline(land, planted)
pd.DataFrame(baseline_audit)


## 4. 三种整数规划

为隔离 HiGHS 原生内存，每个情景由独立工作进程运行。

In [ ]:
from cumcm_lens.cases.crop_planning import run_case
summary = run_case(ROOT / "data/raw", ROOT / "data/processed/2024C")
pd.DataFrame([{k:v for k,v in item.items() if k not in {"audits"}}
              for item in summary["models"]])


## 5. 可行性与最优性审计

In [ ]:
for item in summary["models"]:
    print(item["scenario"], "MIP gap =", item["mip_gap"])
    display(pd.DataFrame(item["audits"]))


## 6. 计划结果

In [ ]:
schedule = pd.read_csv(ROOT / "data/processed/2024C/schedule_robust_worst_case.csv")
display(schedule.head(20))
display(schedule.groupby(["year", "crop_name"])["area_mu"].sum().reset_index().head(30))


## 7. 价格—产量敏感性

In [ ]:
sensitivity = pd.read_csv(ROOT / "data/processed/2024C/sensitivity.csv")
sensitivity.groupby(["price_factor","yield_factor"])["uncapped_profit_yuan"].mean().unstack()


## 8. 结论边界

结果是完整地块单作假设下的经审计可行解；求解器达到时限时须报告 MIP gap，不得写成“已证明最优”。